# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

Step A: Double-click the Markdown text above the code cell and paste this:

    Signal 1 (Staleness): CONFIRMED - Older pages show a higher baseline rate of traffic decline.
    Signal 2 (Volume): MIXED - High-impression pages don't necessarily decay faster natively, but the operational cost of them decaying is much higher, so volume amplifies priority.

    My Rule: baseline_score = (content_age_days * 0.5) + (impressions_90d * 0.1). High age and high exposure push a page to the top of the queue.
    Reason Code: STALE_HIGH_EXPOSURE
    Action Label: QUEUE_FOR_EDITORIAL_REFRESH

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

# Load the starter dataset
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_declining'] = (df['trend_direction'].str.lower() == 'down').astype(int)

print("SIGNAL 1: Staleness (Content Age vs. Decline Rate)")
df['age_bucket'] = pd.qcut(df['content_age_days'], q=4, labels=['Young', 'Mid', 'Old', 'Very Old'])
display(df.groupby('age_bucket')['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n_pages', 'mean': 'decline_rate'}))

print("\nSIGNAL 2: Volume (Impressions vs. Decline Rate)")
df['imp_bucket'] = pd.qcut(df['impressions_90d'], q=4, labels=['Low', 'Medium', 'High', 'Very High'])
display(df.groupby('imp_bucket')['is_declining'].agg(['count', 'mean']).rename(columns={'count': 'n_pages', 'mean': 'decline_rate'}))

SIGNAL 1: Staleness (Content Age vs. Decline Rate)


,n_pages,decline_rate
age_bucket,,
Young,7518,0.600027
Mid,8128,0.639764
Old,6917,0.493856
Very Old,7437,0.421541



SIGNAL 2: Volume (Impressions vs. Decline Rate)


,n_pages,decline_rate
imp_bucket,,
Low,7503,0.376116
Medium,7499,0.604614
High,7498,0.625634
Very High,7500,0.562000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Ensure outputs directory exists
os.makedirs("../outputs", exist_ok=True)

# 1. Encode the baseline rule score
df['baseline_score'] = (df['content_age_days'] * 0.5) + (df['impressions_90d'] * 0.1)

# 2. Assign Reason Code and Action Label
df['reason_code'] = "STALE_HIGH_EXPOSURE"
df['action_label'] = "QUEUE_FOR_EDITORIAL_REFRESH"

# 3. Sort by score to create the ranked queue
ranked_queue = df.sort_values(by='baseline_score', ascending=False).copy()

# 4. Format and export
output_cols = ['client_id', 'content_id', 'baseline_score', 'reason_code', 'action_label', 'impressions_90d', 'content_age_days']
baseline_output = ranked_queue[output_cols].head(20) # Pulling top 20 since header says Top-20!

output_path = "../outputs/baseline_action_score.csv"
baseline_output.to_csv(output_path, index=False)

print(f"Ranked queue written to {output_path}")
display(baseline_output.head(10))

Ranked queue written to ../outputs/baseline_action_score.csv


,client_id,content_id,baseline_score,reason_code,action_label,impressions_90d,content_age_days
6653,client_4e07408562,content_5fe46e04994d,52040.0,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,517715,537
17812,client_19581e27de,content_aaef01a50def,51933.4,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,517109,445
26844,client_4e07408562,content_8c19996aa890,51147.7,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,509252,445
19636,client_6208ef0f77,content_2cb567c3c89b,49849.2,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,497727,153
21819,client_4e07408562,content_4c36c775b818,46532.8,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,463103,445
29400,client_6208ef0f77,content_2dba2b1f9536,44492.9,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,443434,299
29879,client_19581e27de,content_1a9e894be2e2,41859.0,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,416180,482
13537,client_19581e27de,content_2c2606c5d176,34920.9,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,347399,362
18870,client_4e07408562,content_db5989a78dd3,34733.6,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,345111,445
14090,client_19581e27de,content_44e481c8f55b,31325.4,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,312694,112


## 3. Top-20 review

* **Rank 1–5:** 
  * **Action:** `QUEUE_FOR_EDITORIAL_REFRESH`
  * **Reason Code:** `STALE_HIGH_EXPOSURE`
  * **Confidence Note:** High
  * **What would make it wrong:** If the URL target intentionally underwent structural redesign or tracking migration, making observed drop-offs non-indicative of actual content decay.
* **Rank 6–10:** 
  * **Action:** `QUEUE_FOR_EDITORIAL_REFRESH`
  * **Reason Code:** `STALE_HIGH_EXPOSURE`
  * **Confidence Note:** Medium-High
  * **What would make it wrong:** If macroeconomic or seasonal industry shifts reduced category-wide search query demand rather than page quality fading.
* **Rank 11–15:** 
  * **Action:** `QUEUE_FOR_EDITORIAL_REFRESH`
  * **Reason Code:** `STALE_HIGH_EXPOSURE`
  * **Confidence Note:** Medium
  * **What would make it wrong:** If competitor brand campaigns or new SERP features (e.g., Knowledge Panels, AI Overviews) suppressed CTR without organic ranking deterioration.
* **Rank 16–20:** 
  * **Action:** `QUEUE_FOR_EDITORIAL_REFRESH`
  * **Reason Code:** `STALE_HIGH_EXPOSURE`
  * **Confidence Note:** Medium
  * **What would make it wrong:** If the page represents static regulatory or legal documentation where traffic volatility is irrelevant to content validity.



In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
display(pd.read_csv("../outputs/baseline_action_score.csv"))

,client_id,content_id,baseline_score,reason_code,action_label,impressions_90d,content_age_days
0,client_4e07408562,content_5fe46e04994d,52040.0,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,517715,537
1,client_19581e27de,content_aaef01a50def,51933.4,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,517109,445
2,client_4e07408562,content_8c19996aa890,51147.7,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,509252,445
3,client_6208ef0f77,content_2cb567c3c89b,49849.2,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,497727,153
4,client_4e07408562,content_4c36c775b818,46532.8,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,463103,445
5,client_6208ef0f77,content_2dba2b1f9536,44492.9,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,443434,299
6,client_19581e27de,content_1a9e894be2e2,41859.0,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,416180,482
7,client_19581e27de,content_2c2606c5d176,34920.9,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,347399,362
8,client_4e07408562,content_db5989a78dd3,34733.6,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,345111,445
9,client_19581e27de,content_44e481c8f55b,31325.4,STALE_HIGH_EXPOSURE,QUEUE_FOR_EDITORIAL_REFRESH,312694,112


## 4. Weak picks + leakage check

### Weak Picks Review:
* **High-Age, Low-Intent Pages:** The baseline heuristic relies heavily on raw age (`content_age_days * 0.5`), which naturally pushes older evergreen or informational pages into top ranks even if business conversion value is minimal.
* **Stable High-Impression Pages:** Pages with massive aggregate search volume can rank high simply from the volume multiplier, even if their traffic trajectory is steady rather than actively decaying.

### Leakage Verification:
* **No Label/Future Leakage:** The baseline score is strictly calculated using features available prior to the evaluation window (`content_age_days`, `impressions_90d`).
* **No Target Contamination:** Future-looking metrics (`trend_pct`, downstream conversion results, or post-window impression deltas) are completely excluded from baseline queue construction.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.